# B5 · Selecting by attribute: finding the districts you need

> **Time:** about 90 minutes  
> **You need:** Part A, especially A9 · Pandas tables and A10 · Plotting with Matplotlib, and Lessons B1 to B4. You should be able to read a GeoDataFrame with `gpd.read_file` and convert it with `to_crs(32645)`.  
> **By the end you can:**
> 1. use your pandas skills on a **GeoDataFrame**: pick columns and rows, filter with conditions, `.isin` and `.str`,
> 2. add an area column in km² and sort districts by size,
> 3. select features inside a box of coordinates with `.cx`, and show any selection highlighted on a map.

Most map questions start with a selection. Which districts are in Bagmati? Which are larger than 3,000 km²? Which reach north of 28.5°N? In A9 · Pandas tables you filtered peaks and landslides. A GeoDataFrame is a pandas table with a map attached, so every one of those skills works here too. The new part is that each selection is still a map, so you can draw it.

**Counting lines in this lesson:** as in A10, the lines that only set up a plot, `fig, ax = plt.subplots()` and `plt.show()`, are not counted in the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 15 of 21 · [Course home](../README.md)  
← Previous: [B4 · Coordinate reference systems in practice: measuring Nepal in metres](B04_Coordinate_reference_systems.ipynb) · Next: [B6 · Joins and dissolve: names, mismatches and provinces](B06_Joins_and_dissolve.ipynb) →

## 1. A GeoDataFrame is still a table

In B3 you met the **GeoDataFrame**: a pandas table with one extra column, `geometry`, that holds a shape for each row. Everything you learned in A9 still works on it.

`districts.columns` lists the columns, `value_counts()` counts values, `.iloc` picks rows by position and `.loc` by label.

There is one new thing to watch. When you pick several columns, keep `"geometry"` in the list if you still want a map. Without it, the result is a plain pandas **DataFrame**: a table that cannot be drawn as a map. `type()` (Lesson A2) tells you which one you have.

**Predict:** the districts file has 77 districts and 4 columns. What will `districts.shape` print?

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
districts = gpd.read_file(DATA / "nepal_districts.geojson")
print(districts.shape)
districts.head()

In [ ]:
print(districts.columns)
print(districts["province_code"].value_counts())

In [ ]:
names_only = districts[["district", "province_code"]]
with_map = districts[["district", "geometry"]]
print(type(names_only))
print(type(with_map))

In [ ]:
by_name = districts.set_index("district")
by_name.loc["Kaski", "province_code"]

The table has 77 rows and 4 columns: `district`, `district_code`, `province_code` and `geometry`. Koshi (`NP01`) has the most districts, 14, and Madhesh (`NP02`) the fewest, 8.

`names_only` is a `pandas.DataFrame`, because the geometry column was left out. `with_map` is still a `GeoDataFrame`. Kaski is in province `NP04`.

The provinces appear here only as codes. This table shows which name goes with each code. In B6 you will join the names to the map with code instead of reading them from a table.

| Code | Province |
| --- | --- |
| NP01 | Koshi |
| NP02 | Madhesh |
| NP03 | Bagmati |
| NP04 | Gandaki |
| NP05 | Lumbini |
| NP06 | Karnali |
| NP07 | Sudurpashchim |

### Practice

**Q1 · one line:** Show the districts at positions 10 to 14 with `.iloc`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts.iloc[10:15]
```

The slice `10:15` stops before position 15, so you get five rows: Bhojpur, Chitawan, Dadeldhura, Dailekh and Dang. The file lists districts in alphabetical order. Note the spelling **Chitawan**, which follows the source data.
</details>

**Q2 · two lines:** Use `by_name` from the example to print the district code of Kathmandu, then of Mustang.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(by_name.loc["Kathmandu", "district_code"])
print(by_name.loc["Mustang", "district_code"])
```

Kathmandu is `NP0327` and Mustang `NP0445`. The first four characters of a district code are its province code.
</details>

**Q3 · three lines:** Count the districts in each province into `counts`. Print `counts`, then print `counts.sum()` and `len(counts)`. What do the last two numbers mean?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
counts = districts["province_code"].value_counts()
print(counts)
print(counts.sum())
print(len(counts))
```

`counts.sum()` is 77, the number of districts, and `len(counts)` is 7, the number of different province codes. Checking totals like these after every step catches many mistakes early.
</details>

## 2. Filtering districts with a condition

A comparison on a column gives a **boolean mask**, a Series of `True` and `False`, as in A9. Put the mask in square brackets to keep the rows where it is `True`.

`districts[districts["province_code"] == "NP03"]` keeps the districts of Bagmati. The result is a smaller **GeoDataFrame**, so `.plot()` draws it as a map straight away.

`!=` means "not equal". Combine conditions with `&` (and) or `|` (or), with brackets around each condition.

Choosing some features by the values in their table is called **selecting by attribute**. An **attribute** is any non-geometry value of a feature, like its name or province code.

**Predict:** how many districts does Bagmati have? Look back at the counts in Step 1.

In [ ]:
bagmati = districts[districts["province_code"] == "NP03"]
print(len(bagmati))
bagmati.plot(edgecolor="white")
plt.show()

In [ ]:
outside_valley = (districts["province_code"] == "NP03") & (districts["district"] != "Kathmandu")
print(outside_valley.sum())

east = districts[(districts["province_code"] == "NP01") | (districts["province_code"] == "NP02")]
print(len(east))

Bagmati has 13 districts, and the map shows only those 13. The axes are longitude and latitude in degrees, as in B3.

12 Bagmati districts are not called Kathmandu. Koshi and Madhesh together have 22 districts: `|` keeps a row when either condition is true.

### Practice

**Q1 · one line:** Select the districts of Karnali (`NP06`) and print how many there are.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(districts[districts["province_code"] == "NP06"]))
```

Karnali has 10 districts. The inner part makes the mask, the outer brackets select, and `len` counts the rows.
</details>

**Q2 · two lines:** Select the districts of Gandaki (`NP04`) into `gandaki`. Print their names, sorted alphabetically.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gandaki = districts[districts["province_code"] == "NP04"]
print(gandaki["district"].sort_values())
```

Gandaki has 11 districts, from Baglung to Tanahu, including Nawalparasi East. `sort_values()` on one column needs no column name (A9).
</details>

**Q3 · three lines:** Select every district that is **not** in Bagmati and **not** in Gandaki. Print how many there are, then draw them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mask = (districts["province_code"] != "NP03") & (districts["province_code"] != "NP04")
others = districts[mask]
print(len(others))
others.plot(edgecolor="white")
plt.show()
```

77 − 13 − 11 = 53 districts remain. On the map you see two separate blocks, east and west, with a gap in the middle where Bagmati and Gandaki are. `plt.show()` is a plotting line, so it is not counted.
</details>

## 3. Many values at once with `.isin`

To select three districts by name, you could write three conditions joined by `|`. That gets long fast.

`.isin(list)` checks each value against a whole list and gives `True` when the value is in it. `districts["district"].isin(["Kathmandu", "Lalitpur", "Bhaktapur"])` finds the three districts of the Kathmandu Valley.

The `~` sign (tilde) flips a mask: `True` becomes `False` and `False` becomes `True`. Read `~mask` as "not mask". `districts[~mask]` keeps everything that is **not** in the list.

The names in your list must match the table exactly, letter for letter.

In [ ]:
valley_names = ["Kathmandu", "Lalitpur", "Bhaktapur"]
valley = districts[districts["district"].isin(valley_names)]
valley

In [ ]:
west_codes = ["NP06", "NP07"]
in_west = districts["province_code"].isin(west_codes)
print(in_west.sum())
print((~in_west).sum())

The first cell finds all three valley districts, each with its own row label from the original table (9, 35 and 37).

Karnali and Sudurpashchim together have 19 districts, and 58 districts are in the other five provinces. 19 + 58 = 77, so every district was counted once.

### Practice

**Q1 · one line:** Select the four districts `Nawalparasi East`, `Nawalparasi West`, `Rukum East` and `Rukum West` with `.isin`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts[districts["district"].isin(["Nawalparasi East", "Nawalparasi West", "Rukum East", "Rukum West"])]
```

Four rows come back. Older maps had a single Nawalparasi and a single Rukum; each was later split in two, which is why Nepal now has 77 districts.
</details>

**Q2 · two lines:** Use `.isin` and `~` to select every district that is **not** in the Kathmandu Valley list `valley_names` from the example. Print how many there are.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
not_valley = districts[~districts["district"].isin(valley_names)]
print(len(not_valley))
```

74 districts remain: 77 minus the three in the list.
</details>

**Q3 · three lines:** The practice landslide file has records in Dolakha, Rasuwa and Sindhupalchok. Select those three districts, print their names and province codes, and draw them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slide_names = ["Dolakha", "Rasuwa", "Sindhupalchok"]
chosen = districts[districts["district"].isin(slide_names)]
print(chosen[["district", "province_code"]])
chosen.plot(edgecolor="white")
plt.show()
```

All three are in Bagmati (`NP03`), along Nepal's northern border. On the map they form one connected block, because Sindhupalchok sits between Rasuwa and Dolakha.
</details>

## 4. Searching names with `.str`

Sometimes you do not know the exact name. Is it Chitwan or Chitawan? Kavre or Kabhrepalanchok?

In A2 you used string methods like `.upper()` on one string. A text column has the same tools behind `.str`, and they work on every row at once:

| Code | True when the name… |
| --- | --- |
| `.str.startswith("S")` | starts with S |
| `.str.endswith("pur")` | ends with "pur" |
| `.str.contains("Chit")` | contains "Chit" anywhere |
| `.str.contains("chit", case=False)` | contains it, ignoring capitals |

`.str.len()` gives the number of characters in each name, and `.str.upper()` gives the names in capitals.

**Predict:** how many district names start with S?

In [ ]:
starts_s = districts["district"].str.startswith("S")
print(starts_s.sum())
print(districts.loc[starts_s, "district"])

In [ ]:
print(districts.loc[districts["district"].str.contains("chit", case=False), "district"])
print(districts.loc[districts["district"].str.endswith("pur"), "district"])

Eleven names start with S, from Salyan to Syangja. `.loc[mask, "district"]` filters the rows and picks one column, as in A9.

A search for "chit" with `case=False` finds the district even though it is spelled **Chitawan** in this file. Searching for part of a name is a good way to discover how a dataset spells it. Six names end in "pur", such as Bhaktapur and Udayapur.

### Practice

**Q1 · one line:** Show the district whose name contains `Kabhre`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts[districts["district"].str.contains("Kabhre")]
```

One row comes back: Kabhrepalanchok, code `NP0324`, in Bagmati. Many people call it Kavre, which this file does not use.
</details>

**Q2 · two lines:** Add a column `name_length` with the number of characters in each district name. Then show the name and length of the 3 longest names, using `sort_values` and `head`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts["name_length"] = districts["district"].str.len()
districts.sort_values("name_length", ascending=False)[["district", "name_length"]].head(3)
```

Nawalparasi East and Nawalparasi West have 16 characters each, counting the space, followed by Kabhrepalanchok with 15.
</details>

**Q3 · three lines:** Find the district names that contain a space. Print how many there are and the names. What do these districts have in common?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
has_space = districts["district"].str.contains(" ")
print(has_space.sum())
print(districts.loc[has_space, "district"])
```

There are 4: Nawalparasi East, Nawalparasi West, Rukum East and Rukum West, the districts made by splitting two older ones. A space is a character like any other, so `contains(" ")` works.
</details>

## 5. A new column and sorting: the largest districts

To sort districts by size you need an area column. In B4 you learned the rule: never measure area in degrees. Convert to UTM zone 45N (EPSG:32645), whose unit is the metre, then use `.area`.

`districts.to_crs(32645).area` gives one area in m² per district, in the same row order and with the same row labels as `districts`. So you can store it as a new column of `districts`. pandas matches the values by row label, and `districts` itself stays in degrees.

Divide by 1,000,000 to turn m² into km². Then `sort_values` and `head`, from A9, give the largest or the smallest districts.

These areas come from simplified boundaries, so treat them as approximate. All 77 districts add up to about 148,017 km², as in B4. Nepal spans UTM zones 44N and 45N, split at 84°E, so using zone 45N for the whole country stretches the west slightly. That is fine for ranking districts; for precise national areas an equal-area projection is better.

In [ ]:
districts["area_km2"] = (districts.to_crs(32645).area / 1000000).round(1)
print(districts.crs)
print(districts["area_km2"].sum().round())

In [ ]:
largest = districts.sort_values("area_km2", ascending=False)
largest[["district", "province_code", "area_km2"]].head(5)

`districts.crs` is still `EPSG:4326`: the conversion was made on a temporary copy, and only the area numbers were kept. The rounded areas add up to 148,017 km².

Dolpa is the largest district in this data, about 7,965 km², followed by Humla with about 6,039 km². Both are in Karnali (`NP06`).

### Practice

**Q1 · one line:** Show the five smallest districts with their areas.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts.sort_values("area_km2")[["district", "area_km2"]].head(5)
```

Bhaktapur is the smallest, about 122.6 km², followed by Lalitpur, Kathmandu, Parbat and Terhathum. Without `ascending=False`, the smallest come first.
</details>

**Q2 · two lines:** Select the districts larger than 3,000 km² into `big`. Print how many there are and their names.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big = districts[districts["area_km2"] > 3000]
print(len(big), list(big["district"]))
```

11 districts are larger than 3,000 km², including Dolpa, Humla, Gorkha and also Kailali and Dang in the lowlands. `list()` turns the column into a plain Python list, which prints on one line.
</details>

**Q3 · three lines:** Which is the largest district in Bagmati (`NP03`)? Filter, sort by area from largest, and print the name and area of the first row with `.iloc[0]`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati = districts[districts["province_code"] == "NP03"]
top = bagmati.sort_values("area_km2", ascending=False).iloc[0]
print(top["district"], top["area_km2"])
```

Sindhupalchok, about 2,489.7 km², only slightly ahead of Sindhuli at about 2,482.7 km². With simplified boundaries, such a close result could flip, so do not treat the ranking as official.
</details>

## 6. Selecting inside a box of coordinates with `.cx`

So far you selected by attribute. GeoPandas also lets you select by **location** with a box of coordinates: `gdf.cx[x_min:x_max, y_min:y_max]`.

As always, **x comes first, and x is longitude**. `districts.cx[85:86, 27:28]` means longitude 85° to 86°E and latitude 27° to 28°N.

Leave one end of a slice empty to leave that side open. `districts.cx[:, 28.5:]` means "any longitude, latitude from 28.5°N northwards".

A feature is selected if **any part** of it falls in the box. The whole district is kept, not cut at the edge; cutting is called clipping, and comes in B7.

`.cx` works on points too. Below, the landslide CSV becomes a GeoDataFrame with `points_from_xy`, as in B3.

In [ ]:
north = districts.cx[:, 28.5:]
print(len(north))
print(list(north["district"]))

In [ ]:
# Kathmandu Valley box (approximate), from B2
valley_box = districts.cx[85.2:85.55, 27.55:27.82]
print(len(valley_box))
print(list(valley_box["district"]))

In [ ]:
slides_table = pd.read_csv(DATA / "practice_landslides.csv")
slides = gpd.GeoDataFrame(slides_table,
                          geometry=gpd.points_from_xy(slides_table["lon"], slides_table["lat"]),
                          crs="EPSG:4326")
east_slides = slides.cx[86:, :]
print(len(east_slides))
print(east_slides["district"].value_counts())

29 districts reach north of 28.5°N. Some are mountain districts, like Humla and Mustang. Others, like Kailali and Kanchanpur, are in the lowlands of the far west. Western Nepal lies further north than eastern Nepal, so latitude alone does not tell you the height.

The Kathmandu Valley box selects 8 districts, not 3. Districts such as Nuwakot and Makawanpur are kept because a small part of them pokes into the box.

13 landslides are east of 86°E: 9 in Dolakha and 4 in Taplejung.

### Practice

**Q1 · one line:** How many districts reach south of 27°N? Leave the start of the latitude slice open.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
len(districts.cx[:, :27])
```

19 districts reach south of 27°N. They are all in the east, in Koshi, Madhesh and Bagmati (Sindhuli), and include hill districts such as Bhojpur and Ilam. Eastern Nepal lies further south than the west, the mirror image of Step 6's northern selection.
</details>

**Q2 · two lines:** Select the landslides west of 85°E into `west_slides`, then count them per district.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
west_slides = slides.cx[:85, :]
print(west_slides["district"].value_counts())
```

19 landslides are west of 85°E: 9 in Chitwan, 5 in Myagdi, 4 in Gorkha and 1 in Kaski. Here the slice is on x, the longitude, so it comes first.
</details>

**Q3 · three lines:** Select the districts that reach east of 87.5°E. Print how many there are and their names, then draw them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
far_east = districts.cx[87.5:, :]
print(len(far_east), list(far_east["district"]))
far_east.plot(edgecolor="white")
plt.show()
```

8 districts reach east of 87.5°E, from Taplejung on the border with China down to Jhapa in the plains. They are all in Koshi.
</details>

## 7. Showing a selection on a map

A selection on its own floats in empty space. A reader understands it much better against the whole country.

Draw several layers on the **same axes**, as you did with several lines in A10. Create the axes with `fig, ax = plt.subplots()`, then pass `ax=ax` to each `.plot()`.

Layers are drawn in order, so the **last one is on top**. Draw the background first, in a quiet colour such as light grey. Then draw the selection in a strong colour, and points last.

`edgecolor="white"` draws thin white borders between districts. `markersize` sets the size of points. Give the map a title, and label the axes with their units, as in A10.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
north.plot(ax=ax, color="tab:blue", edgecolor="white")
ax.set_title("Districts that reach north of 28.5°N")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
bagmati.plot(ax=ax, color="tab:green", edgecolor="white")
slides.plot(ax=ax, color="black", markersize=10)
ax.set_title("Bagmati province and the practice landslides")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()

In the first map the blue districts cover the whole far west, lowlands included, and the north of the country as far east as Gorkha. East of about 85°E, all of Nepal lies south of 28.5°N.

In the second map, most landslide points sit inside Bagmati. There is a group in Gandaki to the west and a few points in Koshi to the east. These records are made-up practice data.

### Practice

**Q1 · one line:** Draw `north` on its own, coloured by its `province_code` column, with a legend.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
north.plot(column="province_code", legend=True)
```

`column=` colours each district by its value, as in B3. The legend shows four provinces: `NP04`, `NP05`, `NP06` and `NP07`. None of the 29 northern districts is in Koshi, Madhesh or Bagmati.
</details>

**Q2 · two lines:** Highlight the districts of Karnali (`NP06`) in orange over all districts in light grey. Select and draw Karnali in one line.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
districts[districts["province_code"] == "NP06"].plot(ax=ax, color="tab:orange", edgecolor="white")
plt.show()
```

Two lines draw the map; the other two only set it up. A selection is a GeoDataFrame, so `.plot()` can follow the square brackets directly. Karnali fills the north-west of the country, except for the far western strip, which is Sudurpashchim.
</details>

**Q3 · three lines:** Draw all districts in light grey, the Kathmandu Valley districts `valley` from Step 3 in red, and the landslides `slides` as black points. Add a title.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
valley.plot(ax=ax, color="tab:red")
slides.plot(ax=ax, color="black", markersize=10)
ax.set_title("Kathmandu Valley districts and the practice landslides")
plt.show()
```

The three valley districts are tiny at this scale. None of the practice landslides fall in them; the nearest points are in Rasuwa and Sindhupalchok to the north.
</details>

## 8. When it breaks: two conditions without brackets

This cell tries to keep the Bagmati districts other than Kathmandu, but the brackets around each condition are missing. Run it.

In [ ]:
districts[districts["province_code"] == "NP03" & districts["district"] != "Kathmandu"]

Read the last line: `TypeError: unsupported operand type(s) for &: 'str' and 'StringArray'`.

Python does `&` **before** `==` and `!=`. So it first tried `"NP03" & districts["district"]`: a single string "and" a whole column of strings. That makes no sense, hence the `TypeError`.

The fix is the rule from A9: put brackets around every condition.

`districts[(districts["province_code"] == "NP03") & (districts["district"] != "Kathmandu")]`

A second kind of mistake gives **no error at all**: an empty result. `districts.cx[27:28, 85:86]` swaps latitude and longitude and returns 0 rows, because no part of Nepal lies at longitude 27°E. When a selection comes back empty, check the order of x and y and the exact spelling of names first.

**Try it:** fix the cell above. It should return 12 rows.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Bagmati at a glance

Select the districts of Bagmati (`NP03`). Print how many there are, their total area in km² rounded to a whole number, and the names of the smallest and the largest district. Use the `area_km2` column and `sort_values`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati = districts[districts["province_code"] == "NP03"]
print(len(bagmati))
print(round(bagmati["area_km2"].sum()))

ranked = bagmati.sort_values("area_km2")
print(ranked.iloc[0]["district"], ranked.iloc[0]["area_km2"])
print(ranked.iloc[-1]["district"], ranked.iloc[-1]["area_km2"])
```

Bagmati has 13 districts covering about 20,292 km² in this data. The smallest is Bhaktapur (122.6 km²) and the largest Sindhupalchok (2,489.7 km²). After sorting, position 0 is the smallest and position −1 the largest.
</details>

### Review 2 · Area per province

Use `groupby` (A9) to add up `area_km2` for each province code. Print the result sorted from the largest province to the smallest, then print the total of all provinces.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
area_by_province = districts.groupby("province_code")["area_km2"].sum()
area_by_province = area_by_province.sort_values(ascending=False)
print(area_by_province.round())
print(area_by_province.sum().round())
```

Karnali (`NP06`) is the largest province in this data, about 30,856 km², and Madhesh (`NP02`) the smallest, about 9,588 km². The total is again 148,017 km². In B6 you will merge the district shapes themselves into provinces, with `dissolve`.
</details>

### Review 3 · Large and northern

Find the districts that reach north of 28.5°N **and** are larger than 3,000 km². Use `north` from Step 6 and filter it on `area_km2`. Print their names and areas sorted from largest, then highlight them on a map over all districts.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
big_north = north[north["area_km2"] > 3000]
print(big_north.sort_values("area_km2", ascending=False)[["district", "area_km2"]])

fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
big_north.plot(ax=ax, color="tab:purple", edgecolor="white")
ax.set_title("Northern districts larger than 3,000 km²")
plt.show()
```

Seven districts match: Dolpa, Humla, Gorkha, Mustang, Bajhang, Kailali and Mugu. Kailali is a lowland district, but its northern edge reaches past 28.5°N. A `.cx` selection is a GeoDataFrame, so you can filter it by attribute like any other.
</details>

### Review 4 · Which landslide districts are on the map?

Select the districts whose names appear in the `district` column of `slides`, using `.isin(slides["district"])`. Print how many districts you found. The landslide file names 8 districts; do you find 8? Then use `~` to find the landslide records whose district is **not** in `districts["district"]`, and count their district names.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
with_slides = districts[districts["district"].isin(slides["district"])]
print(len(with_slides))
print(list(with_slides["district"]))

unmatched = slides[~slides["district"].isin(districts["district"])]
print(len(unmatched))
print(unmatched["district"].value_counts())
```

Only 7 districts match. The 9 unmatched records all say **Chitwan**, while the boundary file spells it **Chitawan**. Mismatched spellings like this are very common in real data. B6 shows how to detect them systematically and fix them.
</details>

## Recap

- A GeoDataFrame is a pandas table with a `geometry` column, so `.loc`, `.iloc`, `value_counts`, `sort_values` and `groupby` all work. Keep `"geometry"` in a column list if you still want a map.
- Select by attribute with a boolean mask: `districts[districts["province_code"] == "NP03"]`. Use `&` and `|` with brackets around each condition, `.isin(list)` for many values and `~` for "not".
- `.str.contains`, `.str.startswith`, `.str.endswith` and `.str.len` search and measure text in a whole column; `case=False` ignores capitals.
- For areas, convert a copy to UTM 45N: `districts["area_km2"] = districts.to_crs(32645).area / 1000000`.
- `.cx[x_min:x_max, y_min:y_max]` selects features with any part in a box, longitude first. Show a selection by drawing all features in grey first and the selection on top with `ax=ax`.

### Words you learned

| Word | Meaning |
| --- | --- |
| attribute | A non-geometry value of a feature, such as its name or province code |
| selecting by attribute | Choosing features by the values in their table |
| boolean mask | A Series of `True`/`False` values used to keep some rows |
| `.isin(list)` | `True` for each value that appears in the list |
| `~` (tilde) | Flips a mask: "not" |
| `.str` | Gives string methods to a whole text column |
| `.cx` | Selects features that have any part inside a box of x and y coordinates |
| layer | One dataset drawn on a map; later layers are drawn on top |
| `ax=ax` | Tells `.plot()` to draw on an existing axes, so layers share one map |

## Next

**B6 · Joins and dissolve:** add the province names to the map by joining a table, fix the Chitwan/Chitawan mismatch, and merge districts into provinces.

<!-- pygile-nav -->
**PyGILE Course** · lesson 15 of 21 · [Course home](../README.md)  
← Previous: [B4 · Coordinate reference systems in practice: measuring Nepal in metres](B04_Coordinate_reference_systems.ipynb) · Next: [B6 · Joins and dissolve: names, mismatches and provinces](B06_Joins_and_dissolve.ipynb) →